In [1]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess

/Users/ming/Ming/Learning/Projects/llm-learning/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


In [3]:
# Connect to client libraries

openai = OpenAI()
ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(api_key="ollama", base_url=ollama_url)


In [16]:
# models = ["gpt-6.1-sol", "qwen2.5-coder", "deepseek-coder-v2", "gpt-oss:20b"]
models = ["qwen2.5-coder", "deepseek-coder-v2", "gpt-oss:20b"]

# clients = {"gpt-6.1-sol": openai, "qwen2.5-coder": ollama, "deepseek-coder-v2": ollama, "gpt-oss:20b": ollama}
clients = {"qwen2.5-coder": ollama, "deepseek-coder-v2": ollama, "gpt-oss:20b": ollama}


In [17]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Darwin',
  'arch': 'arm64',
  'release': '25.6.0',
  'version': 'Darwin Kernel Version 25.6.0: Fri Jul 31 19:19:08 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T6050',
  'kernel': '25.6.0',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'arm64-apple-darwin25.6.0'},
 'package_managers': ['xcode-select (CLT)'],
 'cpu': {'brand': 'Apple M5 Pro',
  'cores_logical': 15,
  'cores_physical': 15,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'Apple clang version 21.0.0 (clang-2100.3.34.2)',
   'g++': 'Apple clang version 21.0.0 (clang-2100.3.34.2)',
   'clang': 'Apple clang version 21.0.0 (clang-2100.3.34.2)',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': 'GNU Make 3.81'},
  'linkers': {'ld_lld': ''}}}

In [18]:
compile_command = ["clang++", "-std=c++17", "-Ofast", "-mcpu=native", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]

In [19]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [20]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [21]:
def write_output(cpp):
    with open("main.cpp", "w") as f:
        f.write(cpp)

In [22]:
def port(model, python):
    client = clients[model]
    openai_reasoning_models = {"gpt-6.1-sol"}
    reasoning_effort = "high" if model in openai_reasoning_models else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)
    return reply

In [23]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [24]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [25]:
def compile_and_run():
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    except subprocess.CalledProcessError as e:
        print(f"An error occurred:\n{e.stderr}")

In [26]:
with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label="Python code:", lines=28, value=pi)
        cpp = gr.Textbox(label="C++ code:", lines=28)
    with gr.Row():
        model = gr.Dropdown(models, label="Select model", value=models[0])
        convert = gr.Button("Convert code")

    convert.click(port, inputs=[model, python], outputs=[cpp])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7903
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/ming/Ming/Learning/Projects/llm-learning/.venv/lib/python3.13/site-packages/gradio/queueing.py", line 848, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/Users/ming/Ming/Learning/Projects/llm-learning/.venv/lib/python3.13/site-packages/gradio/route_utils.py", line 417, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "/Users/ming/Ming/Learning/Projects/llm-learning/.venv/lib/python3.13/site-packages/gradio/blocks.py", line 2698, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "/Users/ming/Ming/Learning/Projects/llm-learning/.venv/lib/python3.13/site-packages/gradio/blocks.py", line 1963, in call_function
    prediction = await anyio.to

In [27]:
!ollama list

]11;?\NAME                ID              SIZE      MODIFIED     
qwen3.5:27b         64ae5b1d651f    17 GB     3 weeks ago     
gpt-oss:20b         17052f91a42e    13 GB     7 weeks ago     
llama3.2:latest     a80c4f17acd5    2.0 GB    7 weeks ago     
deepseek-r1:1.5b    e0979632db5a    1.1 GB    2 months ago    
gpt-oss:latest      17052f91a42e    13 GB     2 months ago    
phi3:latest         4f2222927938    2.2 GB    2 months ago    
gemma3:270m         e7d36fb2c3b3    291 MB    2 months ago    
qwen3.6:27b         a50eda8ed977    17 GB     2 months ago    


In [28]:
!ollama pull "qwen2.5-coder"

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling 60e05f210007:   0% ▕                  ▏  88 KB/4.7 GB                  pulling manifest 
pulling 60e05f210007:   0% ▕                  ▏ 523 KB/4.7 GB                  pulling manifest 
pulling 60e05f210007:   0% ▕                  ▏ 1.8 MB/4.7 GB                  pulling manifest 
pulling 60e05f210007:   0% ▕                  ▏ 3.2 MB/4.7 GB                  pulling manifest 
pulling 60e05f210007:   0% ▕                  ▏ 3.8 MB/4.7 GB                  pulling manifest 
pulling 60e05f210007:   0% ▕                  ▏ 5.0 MB/4.7 GB                  pulling manifest 
pulling 60e05f210007:   0% ▕                  ▏ 6.5 MB/4.7 GB                  pulling manifest 
pulling 60

In [30]:
port("qwen2.5-coder", pi)

'\n#include <chrono>\n#include <iostream>\n#include <iomanip>\n\ndouble calculate(int iterations, double param1, double param2) {\n    double result = 1.0;\n    for (int i = 1; i <= iterations; ++i) {\n        double j = i * param1 - param2;\n        result -= (1 / j);\n        j = i * param1 + param2;\n        result += (1 / j);\n    }\n    return result;\n}\n\nint main() {\n    auto start_time = std::chrono::high_resolution_clock::now();\n    double result = calculate(200000000, 4, 1) * 4;\n    auto end_time = std::chrono::high_resolution_clock::now();\n\n    std::cout << std::fixed << std::setprecision(12) << "Result: " << result << std::endl;\n    auto duration = std::chrono::duration_cast<std::chrono::seconds>(end_time - start_time);\n    std::cout << "Execution Time: " << duration.count() << " seconds" << std::endl;\n\n    return 0;\n}\n'

port("qwen2.5-coder", pi)

Result: 3.141592656089
Execution Time: 3 seconds

In [31]:
!ollama pull "deepseek-coder-v2"

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling 5ff0abeeac1d:   0% ▕                  ▏ 588 KB/8.9 GB                  pulling manifest 
pulling 5ff0abeeac1d:   0% ▕                  ▏ 1.8 MB/8.9 GB                  pulling manifest 
pulling 5ff0abeeac1d:   0% ▕                  ▏ 2.2 MB/8.9 GB                  pulling manifest 
pulling 5ff0abeeac1d:   0% ▕                  ▏ 3.5 MB/8.9 GB                  pulling manifest 
pulling 5ff0abeeac1d:   0% ▕                  ▏ 4.2 MB/8.9 GB                  pulling manifest 
pulling 5ff0abeeac1d:   0% ▕                  ▏ 6.9 MB/8.9 GB                  pulling manifest 
pulling 5ff0abeeac1d:   0% ▕                  ▏ 7.4 MB/8.9 GB                  pulling manifest 
pulling 5ff0abeeac1d:   0% ▕                  ▏ 7.5 MB/8.9 GB                  pulling

In [ ]:
port("qwen2.5-coder", pi)

In [33]:
port("deepseek-coder-v2", pi)

' \n#include <iostream>\n#include <chrono>\n\ndouble calculate(int iterations, double param1, double param2) {\n    double result = 1.0;\n    for (int i = 1; i <= iterations; ++i) {\n        double j = i * param1 - param2;\n        result -= (1.0 / j);\n        j = i * param1 + param2;\n        result += (1.0 / j);\n    }\n    return result;\n}\n\nint main() {\n    auto start_time = std::chrono::high_resolution_clock::now();\n    double result = calculate(200000000, 4.0, 1.0) * 4.0;\n    auto end_time = std::chrono::high_resolution_clock::now();\n    auto duration = std::chrono::duration_cast<std::chrono::microseconds>(end_time - start_time).count();\n\n    std::cout.precision(12);\n    std::cout << "Result: " << result << std::endl;\n    std::cout << "Execution Time: " << static_cast<double>(duration) / 1000000.0 << " seconds" << std::endl;\n\n    return 0;\n}\n'

port("deepseek-coder-v2", pi)

Result: 3.14159265609
Execution Time: 1.527321 seconds

In [34]:
!ollama pull "gpt-oss:20b"

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling 8ecb5d4ad617:   0% ▕                  ▏ 156 KB/ 13 GB                  pulling manifest 
pulling 8ecb5d4ad617:   0% ▕                  ▏ 337 KB/ 13 GB                  pulling manifest 
pulling 8ecb5d4ad617:   0% ▕                  ▏ 1.3 MB/ 13 GB                  pulling manifest 
pulling 8ecb5d4ad617:   0% ▕                  ▏ 1.5 MB/ 13 GB                  pulling manifest 
pulling 8ecb5d4ad617:   0% ▕                  ▏ 1.8 MB/ 13 GB                  pulling manifest 
pulling 8ecb5d4ad617:   0% ▕                  ▏ 2.7 MB/ 13 GB                  pulling manifest 
pulling 8ecb5d4ad617:   0% ▕                  ▏ 3.1 MB/ 13 GB                  pulling manifest 
pulling 8ecb5d4ad617:   0% ▕                  ▏ 3.7 MB/ 13 GB                  pulling

In [35]:
port("gpt-oss:20b", pi)

'#include <iostream>\n#include <chrono>\n#include <iomanip>\n\nint main() {\n    const unsigned long long iterations = 200\'000\'000ULL;\n    const double param1 = 4.0;\n    const double param2 = 1.0;\n\n    auto start = std::chrono::high_resolution_clock::now();\n\n    double result = 1.0;\n    for (unsigned long long i = 1; i <= iterations; ++i) {\n        double j = i * param1 - param2;\n        result -= 1.0 / j;\n        j = i * param1 + param2;\n        result += 1.0 / j;\n    }\n    result *= 4.0;\n\n    auto end = std::chrono::high_resolution_clock::now();\n    std::chrono::duration<double> elapsed = end - start;\n\n    std::cout << std::fixed << std::setprecision(12)\n              << "Result: " << result << \'\\n\'\n              << std::fixed << std::setprecision(6)\n              << "Execution Time: " << elapsed.count() << " seconds\\n";\n\n    return 0;\n}\n'

port("gpt-oss:20b", pi)

Result: 3.141592656089
Execution Time: 3.138000 seconds
